# Comparaison de modèles IA


COPIER SA POUR METRE VOS IMAGE 

<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/cnn.jpg" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure n : LEGENDE
    </div>
</div>

Ce notebook est dédié à l'analyse des nos différents modèles. 
Pour rappel, nos base de données sont les suivantes :
 - "Municipal Waste Management Cost Prediction"
 - "Garbage Classification"
Et notre problématique est Comment l’intelligence artificielle peut-elle soutenir une gestion intelligente des déchets en combinant la prévision des flux de déchets communaux et l’assistance au tri pour les citoyens à partir d’images ?

L'objectif est de comparer nos différents modèles afin de savoir lequel est le plus performant.

##  Table des Matières
1. [Librairies et Configuration](#Librairies)
2. [Analyse des modèles IA de la Base de Données : Municipal Waste Management Cost Prediction](#bd1)
3. [Analyse des modèles IA de la Base de Données : Garbage Classification](#bd2)
4. [Conclusion](#conclu)

---

<a id="Librairies"></a>
## 1. Librairies et Configuration

In [ ]:
import pandas as pd
import random

import seaborn as sns
import matplotlib.pyplot as plt 
import matplotlib.image as mpimg
import geopandas as gpd

import os

!pip install rpy2
%load_ext rpy2.ipython


file_path = "../data/public_data_waste_fee.csv"
df = pd.read_csv(file_path)

<a id="bd1"></a>
## 2. Analyse des modèles IA de la Base de Données : Municipal Waste Management Cost Prediction

&nbsp;&nbsp;&nbsp;&nbsp; **a) Régression Linéaire Simple**

&nbsp;&nbsp;&nbsp;&nbsp; **b) Régression Logistique Multinomiale**

In [30]:
%%R
library(nnet)
set.seed(123)


df <- read.csv("../data/public_data_waste_fee.csv")
dechets_colonnes <- c("organic", "paper", "glass", "wood", "metal", "plastic", "raee", "texile", "other")

df[dechets_colonnes][is.na(df[dechets_colonnes])] <- 0

index <- sample(1:nrow(df), size = 0.6 * nrow(df))
train_set <- df[index, ]
test_set  <- df[-index, ]

Y_train <- as.matrix(train_set[, dechets_colonnes])
 
modele_dechets <- multinom(Y_train ~ log(pop) + gdp + wage + alt + urb, 
                           data = train_set, 
                           maxit = 500)

summary(modele_dechets)

predictions <- predict(modele_dechets, newdata = test_set)

vrai_label <- dechets_colonnes[max.col(test_set[, dechets_colonnes])]
accuracy <- sum(diag(table_matrice)) / sum(table_matrice)

print(table_matrice)
print(paste("Précision globale :", round(accuracy * 100, 2), "%"))



# weights:  63 (48 variable)
initial  value 311679.999662 
iter  10 value 278090.206068
iter  20 value 268253.038539
iter  30 value 267645.535109
iter  40 value 262816.587338
iter  50 value 257585.764200
final  value 257565.112017 
converged
Error in (function (expr, envir = parent.frame(), enclos = if (is.list(envir) ||  : 
  object 'table_matrice' not found


RInterpreterError: Failed to parse and evaluate line 'library(nnet)\nset.seed(123)\n\n\ndf <- read.csv("../data/public_data_waste_fee.csv")\ndechets_colonnes <- c("organic", "paper", "glass", "wood", "metal", "plastic", "raee", "texile", "other")\n\ndf[dechets_colonnes][is.na(df[dechets_colonnes])] <- 0\n\nindex <- sample(1:nrow(df), size = 0.6 * nrow(df))\ntrain_set <- df[index, ]\ntest_set  <- df[-index, ]\n\nY_train <- as.matrix(train_set[, dechets_colonnes])\n\nmodele_dechets <- multinom(Y_train ~ log(pop) + gdp + wage + alt + urb, \n                           data = train_set, \n                           maxit = 500)\n\nsummary(modele_dechets)\n\npredictions <- predict(modele_dechets, newdata = test_set)\n\nvrai_label <- dechets_colonnes[max.col(test_set[, dechets_colonnes])]\naccuracy <- sum(diag(table_matrice)) / sum(table_matrice)\n\nprint(table_matrice)\nprint(paste("Précision globale :", round(accuracy * 100, 2), "%"))\n\n'.
R error message: "Error in (function (expr, envir 

Pour la régression logistique multinomiale, nous avons essayé de prédire le taux de déchets par catégorie, avec le niveau de revenus par habitants (gdp), la densité de la population (pden), le nombre d’habitant (pop), l'altitude (alt) et l’indice d’urbanisation (urb). 
Cependant, la matrice de confusion nous montre que le modèle prédit beaucoup la catégorie organic.
Malgré une modification de la répartition des données (60/40), la matrice de confusion révèle que le modèle multinomial reste fortement biaisé en faveur de la catégorie Organic. Ce phénomène d'écrasement s'explique par le déséquilibre natif du dataset (classe organique ultra-majoritaire)

&nbsp;&nbsp;&nbsp;&nbsp; **c) Random Forest**

<a id="bd2"></a>
## 3. Analyse des modèles IA de la Base de Données : Garbage Classification

<div style="margin-left: 30px; font-size: 25px;">
  3.1. Introduction
</div>

Pour préduire le type de dechets a partir d'une image, nous avons décider de creer et d'entrainer un réseaux de neuronne convolutif a partir de notre base de données d'images. Les réseauxx de neuronnes convolutivif dont Yann LeCun deveuloppa le premier réseaux fonctionnels (1998), provient des travaux de D. H. Hubel et T. N. Wiesel (1968) sur le cortex visuel du chat et des primates. Révolutionner par le model Alexnet en 2012, les réseaux de neurone convolutif sont aujourd'hui devenue la methode principale en classifaction d'image.

Pour préduire le type de dechets a partir d'une image, nous avons décider de creer et d'entrainer un réseaux de neuronne
 convolutif (CNN) a partir de notre base de données d'images. Les réseauxx de neuronnes convolutivif dont Yann LeCun
  deveuloppa le premier réseaux fonctionnels (1998), provient des travaux de D. H. Hubel et T. N. Wiesel (1968) 
  sur le cortex visuel du chat et des primates. Révolutionner par le model Alexnet en 2012,
 les réseaux de neurone convolutif sont aujourd'hui devenue la methode principale en classifaction d'image.

 Leur plus gros avantage aprenne eux meme eux-mêmes à identifier les caractéristiques importantes des images via les coucher
 Les réseaux de neurones convolutif sont également invariant à la transalation et conserve les relations spatiales. 

 Pour résumer, les réseaux de neurones convolutifs se composent de 3 types de couches : 
Une couche de convolution qui agit comme un filtre sur l’image pour extraire des caractéristiques (ligne verticale, horizontale, couleurs) via des opérations matricielles.
Une couche de pooling qui réduit la taille des images tout en gardant les informations importantes pour accélérer le traitement. On réduit les zones en prenant généralement la moyenne des pixels ou le maximum d’une zone donnée.
Une couche totalement connectée qui prend les caractéristiques extraites par les couches précédentes. Et donne la décision finale de la prédiction.




<style>
    .img-container {
        text-align: center;
        margin: 20px 0;
    }
    .custom-img {
        width: 600px;
        border: 1px solid #ddd;
        border-radius: 8px;
        box-shadow: 2px 2px 10px rgba(0,0,0,0.1);
    }
    .legende {
        font-style: italic;
        font-size: 0.9em;
        color: #555;
        margin-top: 8px;
    }
</style>

<div style="text-align: center; margin: 20px 0;">
    <img src="imagesNotebook/cnn.jpg" 
         style="width: 600px; border: 1px solid #ddd; border-radius: 8px; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <div style="font-style: italic; font-size: 0.9em; color: #555; margin-top: 8px;">
        Figure 1 : HAQUE, Kh Nafizul. "What is Convolutional Neural Network — CNN (Deep Learning)", LinkedIn, 3 avril 2023
    </div>
</div>

Un CNN fonctionne en deux étapes. Premièrement il enchaîne des couches de convolution et de pooling pour réduire les données. Puis ces informations sont aplaties et transmises à la couche totalement connectée qui prend la décision finale en renvoyant une probabilité.

Pour notre projet nous possédons déjà une base de données avec des images à une taille identique (512 x 384). Nous avons également deux dossiers distincts : un avec des images pour entraîner le modèle et l’autre avec d’autres images pour tester le modèle.

EXPLICATION METHODE

SEED

COLLAB

<div style="margin-left: 30px; font-size: 25px;">
  3.2. Méthodologie
</div>

<div style="margin-left: 30px; font-size: 25px;">
  3.3. Description des Architectures
</div>

&nbsp;&nbsp;&nbsp;&nbsp; **a) Modèle avec couches en décroissant**

&nbsp;&nbsp;&nbsp;&nbsp; **b) Modèle avec couches en croissant**

&nbsp;&nbsp;&nbsp;&nbsp; **c) Modèle AlexNet**

&nbsp;&nbsp;&nbsp;&nbsp; **d) Modèle vggNet**

&nbsp;&nbsp;&nbsp;&nbsp; **e) Modèle Optimisé**


<div style="margin-left: 30px; font-size: 25px;">
  3.4. Résultats et Analyse
</div>

<div style="margin-left: 30px; font-size: 25px;">
  3.5. Discussion / Conclusion
</div>

<a id="conclu"></a>
## 4. Conclusion